MongoDB connection

In [1]:
from pymongo import MongoClient

In [2]:
client = MongoClient("mongodb://localhost:27017/")

In [3]:
db = client["ecommerce_recommendation"]

In [25]:
interactions_collection = db["interactions"]

In [30]:
reviews_collection = db["reviews"]

Loading interactions into pandas

In [5]:
import pandas as pd

In [26]:
interactions = list(interactions_collection.find())

In [32]:
reviews = pd.DataFrame(list(reviews_collection.find()))

In [ ]:
df = pd.DataFrame(interactions)

Keeping only meaningful interactions

In [8]:
df = df[
    df["action"].isin(
        ["purchase", "add_to_cart"]
    )
]

Creating User-Product Matrix

In [9]:
user_product_matrix = df.pivot_table(

    index="user_id",

    columns="product_id",

    values="rating",

    fill_value=0
)

Computing similarity between users

In [10]:
from sklearn.metrics.pairwise import cosine_similarity

In [11]:
user_similarity = cosine_similarity(
    user_product_matrix
)

In [12]:
similarity_df = pd.DataFrame(

    user_similarity,

    index=user_product_matrix.index,

    columns=user_product_matrix.index
)

In [22]:
def recommend_products(user_id, top_n=5):

    similar_users = similarity_df[user_id] \
        .sort_values(ascending=False)

    similar_users = similar_users.iloc[1:6]

    recommended_products = set()

    for similar_user in similar_users.index:

        user_products = df[
            df["user_id"] == similar_user
        ]["product_id"]

        recommended_products.update(user_products)

    current_user_products = set(

        df[
            df["user_id"] == user_id
        ]["product_id"]
    )

    recommendations = recommended_products - current_user_products

    return list(recommendations)[:top_n]

In [23]:
df["user_id"].iloc[0]

'AO94DHGC771SJ'

In [24]:
recommend_products("AO94DHGC771SJ")

['B00005BC0J', 'B00000K135', 'B000067V62', 'B00004ZCJE', 'B000023VW2']

------------------------------------------------------------------------------------------------------------------------------------

AI-Sentiment

In [28]:
from transformers import pipeline

sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model="distilbert/distilbert-base-uncased-finetuned-sst-2-english"
)

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 763.65it/s]


In [33]:
sample = reviews["reviewText"].iloc[0]

result = sentiment_pipeline(sample[:512])

print(result)

[{'label': 'POSITIVE', 'score': 0.9782578349113464}]


Testing on small subset

In [34]:
reviews_subset = reviews.head(100).copy()

AI Sentiment Function

In [35]:
def get_ai_sentiment(text):
    
    try:
        result = sentiment_pipeline(str(text)[:512])[0]

        return pd.Series([
            result["label"],
            result["score"]
        ])

    except:
        return pd.Series([
            "ERROR",
            0
        ])

In [36]:
reviews_subset[
    ["ai_sentiment", "ai_sentiment_score"]
] = reviews_subset["reviewText"].apply(
    get_ai_sentiment
)

In [41]:
reviews_subset["sentiment"] = reviews_subset["analytics"].apply(
    lambda x: x.get("sentiment") if isinstance(x, dict) else None
)

In [42]:
reviews_subset[
    [
        "reviewText",
        "sentiment",
        "ai_sentiment",
        "ai_sentiment_score"
    ]
].head()

,reviewText,sentiment,ai_sentiment,ai_sentiment_score
0,We got this GPS for my husband who is an (OTR)...,positive,positive,0.978258
1,"I'm a professional OTR truck driver, and I bou...",negative,negative,0.998310
2,"Well, what can I say. I've had this unit in m...",neutral,negative,0.975643
3,"Not going to write a long review, even thought...",negative,negative,0.974845
4,I've had mine for a year and here's what we go...,negative,negative,0.999722


In [39]:
reviews_subset["ai_sentiment"] = (
    reviews_subset["ai_sentiment"]
    .str.lower()
)

Applying to larger chunks

In [43]:
batch_size = 500
all_reviews = reviews.copy()

for i in range(0, len(all_reviews), batch_size):
    batch = all_reviews.iloc[i:i+batch_size].copy()

    batch[["ai_sentiment", "ai_sentiment_score"]] = batch["reviewText"].apply(
        get_ai_sentiment
    )

    # update main dataframe
    all_reviews.loc[batch.index, ["ai_sentiment", "ai_sentiment_score"]] = \
        batch[["ai_sentiment", "ai_sentiment_score"]]

    print(f"Processed batch {i} → {i+batch_size}")

Processed batch 0 → 500
Processed batch 500 → 1000
Processed batch 1000 → 1500
Processed batch 1500 → 2000
Processed batch 2000 → 2500
Processed batch 2500 → 3000
Processed batch 3000 → 3500
Processed batch 3500 → 4000
Processed batch 4000 → 4500
Processed batch 4500 → 5000
Processed batch 5000 → 5500
Processed batch 5500 → 6000
Processed batch 6000 → 6500
Processed batch 6500 → 7000
Processed batch 7000 → 7500
Processed batch 7500 → 8000
Processed batch 8000 → 8500
Processed batch 8500 → 9000
Processed batch 9000 → 9500
Processed batch 9500 → 10000
Processed batch 10000 → 10500
Processed batch 10500 → 11000
Processed batch 11000 → 11500
Processed batch 11500 → 12000
Processed batch 12000 → 12500
Processed batch 12500 → 13000
Processed batch 13000 → 13500
Processed batch 13500 → 14000
Processed batch 14000 → 14500
Processed batch 14500 → 15000
Processed batch 15000 → 15500
Processed batch 15500 → 16000
Processed batch 16000 → 16500
Processed batch 16500 → 17000
Processed batch 17000 → 

In [44]:
for _, row in all_reviews.iterrows():

    reviews_collection.update_one(
        {"_id": row["_id"]},
        {
            "$set": {
                "ai_sentiment": row["ai_sentiment"],
                "ai_sentiment_score": float(row["ai_sentiment_score"])
            }
        }
    )

In [45]:
reviews_collection.find_one()

{'_id': ObjectId('6a005895cd2ac99da8b69712'),
 'reviewerID': 'AO94DHGC771SJ',
 'asin': '0528881469',
 'helpful': [0, 0],
 'reviewText': 'We got this GPS for my husband who is an (OTR) over the road trucker.  Very Impressed with the shipping time, it arrived a few days earlier than expected...  within a week of use however it started freezing up... could of just been a glitch in that unit.  Worked great when it worked!  Will work great for the normal person as well but does have the "trucker" option. (the big truck routes - tells you when a scale is coming up ect...)  Love the bigger screen, the ease of use, the ease of putting addresses into memory.  Nothing really bad to say about the unit with the exception of it freezing which is probably one in a million and that\'s just my luck.  I contacted the seller and within minutes of my email I received a email back with instructions for an exchange! VERY impressed all the way around!',
 'overall': 5,
 'summary': 'Gotta have GPS!',
 'unixRe

AI Analytics

Sentiment distribution

In [46]:
df = pd.DataFrame(list(reviews_collection.find()))

df["ai_sentiment"].value_counts()

ai_sentiment
POSITIVE    23579
NEGATIVE    16363
Name: count, dtype: int64

Product reputation

In [47]:
df.groupby("asin")["ai_sentiment_score"].mean().sort_values(ascending=False)

asin
B00006GF1G    0.999676
B00005NPOB    0.999609
B00004WZON    0.999423
B00000JBIA    0.999329
B00006686C    0.999238
                ...   
B00005N5WV    0.809896
B000056I6K    0.803650
B00005I9PF    0.794045
B00005AY7P    0.766210
B00005T3BF    0.731712
Name: ai_sentiment_score, Length: 1616, dtype: float64

Using in recommendation system

In [49]:
interactions = pd.DataFrame(
    list(interactions_collection.find())
)

reviews_df = pd.DataFrame(
    list(reviews_collection.find())
)

In [50]:
reviews_df = reviews_df[
    [
        "reviewerID",
        "asin",
        "ai_sentiment",
        "ai_sentiment_score"
    ]
]

Merging Reviews with Interactions

In [51]:
merged_df = interactions.merge(
    reviews_df,
    left_on=["user_id", "product_id"],
    right_on=["reviewerID", "asin"],
    how="left"
)

Creating Sentiment Weight

In [52]:
def sentiment_weight(sentiment):

    if sentiment == "positive":
        return 1.2

    elif sentiment == "negative":
        return 0.7

    else:
        return 1

In [53]:
merged_df["sentiment_weight"] = (
    merged_df["ai_sentiment"]
    .apply(sentiment_weight)
)

Creating final recommendation score

In [54]:
merged_df["final_score"] = (
    merged_df["rating"] *
    merged_df["sentiment_weight"] *
    merged_df["ai_sentiment_score"]
)

Building new pivot table

In [55]:
user_product_matrix = merged_df.pivot_table(
    index="user_id",
    columns="product_id",
    values="final_score",
    fill_value=0
)

Recalculating similarity

In [56]:
from sklearn.metrics.pairwise import cosine_similarity

Using Top 1000 active users

In [58]:
active_users = (
    merged_df["user_id"]
    .value_counts()
    .head(1000)
    .index
)

In [59]:
filtered_df = merged_df[
    merged_df["user_id"].isin(active_users)
]

In [60]:
ai_user_product_matrix = filtered_df.pivot_table(
    index="user_id",
    columns="product_id",
    values="final_score",
    fill_value=0
)

In [63]:
user_similarity = cosine_similarity(
    ai_user_product_matrix
)

In [64]:
similarity_df = pd.DataFrame(
    user_similarity,
    index=ai_user_product_matrix.index,
    columns=ai_user_product_matrix.index
)

In [65]:
def recommend_products(user_id, top_n=5):

    similar_users = similarity_df[user_id] \
        .sort_values(ascending=False)

    similar_users = similar_users.iloc[1:6]

    recommended_products = set()

    for similar_user in similar_users.index:

        user_products = merged_df[
            merged_df["user_id"] == similar_user
        ]["product_id"]

        recommended_products.update(user_products)

    return list(recommended_products)[:top_n]